In [46]:
import os 
from dotenv import load_dotenv

load_dotenv()

True

In [47]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/telecom_knowledge_base.txt")
documents = loader.load()

print(f"Loaded {len(documents)} document(s) from the TXT file")

print("\n--- First document preview (first 500 chars) ---")
print(documents[0].page_content[:500])

Loaded 1 document(s) from the TXT file

--- First document preview (first 500 chars) ---
# Telecom Knowledge Base

## Purpose
This document is a generic knowledge base for a Retrieval-Augmented Generation (RAG) telecom customer-support chatbot. It uses fictional/generic telecom policies so the chatbot can answer questions from retrieved context without relying on a real carrier's current policies.

---

# 1. Mobile Service Basics

A mobile telecom service normally provides some combination of:
- Mobile voice calling
- SMS messaging
- Mobile data
- 4G/LTE and 5G connectivity
- Intern


In [48]:
from langchain_core.documents import Document

chunks = [
    Document(page_content=chunk)
    for chunk in documents[0].page_content.split("\n\n")
]

print("Number of chunks:", len(chunks))

Number of chunks: 136


In [49]:
print(chunks[1].page_content)

## Purpose
This document is a generic knowledge base for a Retrieval-Augmented Generation (RAG) telecom customer-support chatbot. It uses fictional/generic telecom policies so the chatbot can answer questions from retrieved context without relying on a real carrier's current policies.


In [50]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = Chroma.from_documents(
    chunks,
    embeddings
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7184.18it/s]


In [51]:
print("Chunks:", len(chunks))
print("Vectors in Chroma:", vector_store._collection.count())

Chunks: 136
Vectors in Chroma: 670


In [52]:
query = "Why is my mobile internet slow?"

results = vector_store.similarity_search(
    query,
    k=3
)

print(f"Query: {query}\n")

for i, result in enumerate(results, 1):
    print(f"--- Result {i} ---")
    print(result.page_content)
    print()

Query: Why is my mobile internet slow?

--- Result 1 ---
### Q: Why is my mobile internet slow?
Possible causes include weak signal, network congestion, limited coverage, device settings, or exhausted high-speed data allowance.

--- Result 2 ---
### Q: Why is my mobile internet slow?

--- Result 3 ---
### Q: Why is my mobile internet slow?



In [53]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


# System prompt
SYSTEM_PROMPT = """
You are a helpful telecom assistant.

Answer the question using ONLY the context provided below.

If the context does not contain enough information to answer the question,
say clearly that the information is not available in the provided context.

Context:
{context}
"""


# Prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}")
])


# Retriever from ChromaDB
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)


# Convert retrieved Documents into plain text
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# LLM via Groq
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)


# RAG chain
chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)


print("RAG chain assembled")

RAG chain assembled


In [54]:
import os
from groq import Groq

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

models = client.models.list()

for model in models.data:
    if model.active:
        print(model.id)

canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-22m
meta-llama/llama-prompt-guard-2-86m
allam-2-7b
openai/gpt-oss-safeguard-20b
openai/gpt-oss-120b
qwen/qwen3.8-27b
qwen/qwen3.6-27b
canopylabs/orpheus-arabic-saudi
groq/compound-mini
whisper-large-v3
openai/gpt-oss-20b
whisper-large-v3-turbo
groq/compound


In [55]:
question = "Why is my mobile internet slow?"

print(f"Question: {question}\n")
print("Answer:", chain.invoke(question))

Question: Why is my mobile internet slow?

Answer: Possible causes include weak signal, network congestion, limited coverage, device settings, or exhausted high‑speed data allowance.


In [56]:
question = "Why is my mobile internet slow?"

results = retriever.invoke(question)

for i, doc in enumerate(results, 1):
    print(f"\n--- Retrieved Chunk {i} ---")
    print(doc.page_content)


--- Retrieved Chunk 1 ---
### Q: Why is my mobile internet slow?
Possible causes include weak signal, network congestion, limited coverage, device settings, or exhausted high-speed data allowance.

--- Retrieved Chunk 2 ---
### Q: Why is my mobile internet slow?

--- Retrieved Chunk 3 ---
### Q: Why is my mobile internet slow?


In [57]:
question = "Why can a strong signal still result in slow internet?"

print(f"Question: {question}\n")
print("Answer:", chain.invoke(question))

Question: Why can a strong signal still result in slow internet?

Answer: A strong signal only tells you that the device can receive the network’s radio signal well. It doesn’t guarantee that the network can deliver data quickly. Even with a good signal, the actual internet speed can be limited by factors such as:

* **Congestion** – many users sharing the same cell tower or Wi‑Fi channel can slow down each other’s throughput.  
* **Available network capacity** – the back‑haul or core network may not have enough bandwidth to support high speeds for all users at once.

So, while a strong signal is necessary, it’s not sufficient; congestion and overall network capacity can still result in slow internet.
